# Predictive Modeling & Calibrated Stacking Ensemble
This notebook implements a leak-free Machine Learning modeling pipeline. We use purged and embargoed walk-forward cross-validation splits, train multiple machine learning models (XGBoost, LightGBM, CatBoost), construct a Stacking Ensemble with a Logistic Regression meta-learner, and apply probability calibration.

In [1]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
import joblib
import matplotlib.pyplot as plt
import sys

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, brier_score_loss

sys.path.append('../')
from src.feature_engineering import build_purged_walkforward_splits

ROOT = Path.cwd().parent
CSV = ROOT / 'csv_files'
OUT = ROOT / 'outputs'
ART = ROOT / 'model_artifacts'
ART.mkdir(exist_ok=True)

SEED = 42
np.random.seed(SEED)

## Load Feature Matrix & Define Variable Roles

In [2]:
df = pd.read_csv(ROOT / 'data' / 'historical_with_features.csv')
df['timestamp_utc'] = pd.to_datetime(df['timestamp_utc'])
df = df.sort_values('timestamp_utc').reset_index(drop=True)

num_cols = [
    'Execution Price', 'Size Tokens', 'Size USD', 'Start Position', 'Fee', 'fg_value', 'sent_regime_ord',
    'size_usd_log', 'fee_abs_log', 'exec_price_log', 'hour', 'dow', 'month',
    'acc_wr5', 'acc_wr10', 'acc_wr20', 'acc_wr50', 'acc_wr100',
    'acc_pnl_mean_10', 'acc_pnl_std_10', 'acc_pnl_velocity_10',
    'acc_pnl_mean_20', 'acc_pnl_std_20', 'acc_pnl_velocity_20',
    'acc_pnl_mean_50', 'acc_pnl_std_50', 'acc_pnl_velocity_50',
    'acc_size_mean_10', 'acc_size_mean_20', 'acc_size_mean_50',
    'acc_taker_rate_10', 'acc_taker_rate_20', 'acc_taker_rate_50',
    'acc_streak', 'coin_wr30', 'coin_pnl30', 'coin_taker_rate_30',
    'fg_l1', 'fg_l3', 'fg_l7', 'fg_d1', 'fg_d3', 'fg_r7m', 'fg_r7s',
    'sent_x_taker', 'sent_x_accwr', 'sent_x_accsize', 'sent_x_coinwr', 'size_x_sent'
]
cat_cols = ['Account', 'Coin', 'Side', 'Direction', 'fg_classification']
features = [c for c in num_cols + cat_cols if c in df.columns]

print(f"Total input modeling features: {len(features)}")

Total input modeling features: 54


## Define Purged & Embargoed walk-forward CV splits
We build 5 outer folds chronologically, leaving the last 15% as a final holdout dataset.

In [3]:
n = len(df)
train_val_end = int(n * 0.85)

train_val_df = df.iloc[:train_val_end].copy()
hold_df = df.iloc[train_val_end:].copy()
hold_df.to_csv(ROOT / 'data' / 'holdout_data.csv', index=False)

folds = build_purged_walkforward_splits(train_val_df, n_splits=5, purge_days=1, embargo_days=1)
print(f"Holdout size: {len(hold_df)} rows")
for f, tr, te in folds:
    print(f"Fold {f} - Train size: {len(tr)}, Val size: {len(te)}")

Holdout size: 31662 rows
Fold 1 - Train size: 28521, Val size: 29902
Fold 2 - Train size: 59499, Val size: 29902
Fold 3 - Train size: 88130, Val size: 29902
Fold 4 - Train size: 117669, Val size: 29902
Fold 5 - Train size: 148374, Val size: 29902


## Train Candidate Stack (XGBoost, LightGBM, CatBoost) & Stacking Ensemble
We initialize ColumnTransformer for data processing and build an ensembled stack.

In [4]:
pre = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median'))]), [c for c in num_cols if c in features]),
    ('cat', Pipeline([
        ('imp', SimpleImputer(strategy='most_frequent')),
        ('enc', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
    ]), [c for c in cat_cols if c in features]),
])

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

xgb = Pipeline([('pre', pre), ('clf', XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6, random_state=SEED, n_jobs=-1))])
lgbm = Pipeline([('pre', pre), ('clf', LGBMClassifier(n_estimators=400, learning_rate=0.04, num_leaves=31, random_state=SEED, n_jobs=-1))])
cat = Pipeline([('pre', pre), ('clf', CatBoostClassifier(iterations=300, depth=6, learning_rate=0.05, verbose=0, random_seed=SEED))])

models = {'xgb': xgb, 'lgbm': lgbm, 'cat': cat}
oof_probs = {k: np.zeros(len(train_val_df)) for k in models.keys()}
fold_records = []

X_all = train_val_df[features]
y_all = train_val_df['y'].values

for f, tr, te in folds:
    Xtr, ytr = X_all.iloc[tr], y_all[tr]
    Xte, yte = X_all.iloc[te], y_all[te]
    
    for name, m in models.items():
        m.fit(Xtr, ytr)
        p = m.predict_proba(Xte)[:, 1]
        oof_probs[name][te] = p
        
        auc = roc_auc_score(yte, p) if len(np.unique(yte)) > 1 else np.nan
        pr = average_precision_score(yte, p)
        fold_records.append({'model': name, 'fold': f, 'auc': auc, 'pr_auc': pr})

scores_df = pd.DataFrame(fold_records)
print("Fold validation metrics:")
scores_df.groupby('model')[['auc', 'pr_auc']].mean()

[LightGBM] [Info] Number of positive: 11278, number of negative: 17243
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002418 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 8119
[LightGBM] [Info] Number of data points in the train set: 28521, number of used features: 54
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.395428 -> initscore=-0.424552
[LightGBM] [Info] Start training from score -0.424552


/home/manas/Documents/assignments/ds/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 25715, number of negative: 33784
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003277 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 8166
[LightGBM] [Info] Number of data points in the train set: 59499, number of used features: 54
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.432192 -> initscore=-0.272913
[LightGBM] [Info] Start training from score -0.272913


/home/manas/Documents/assignments/ds/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 37265, number of negative: 50865
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.006454 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 8187
[LightGBM] [Info] Number of data points in the train set: 88130, number of used features: 54
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.422841 -> initscore=-0.311121
[LightGBM] [Info] Start training from score -0.311121


...

/home/manas/Documents/assignments/ds/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 63651, number of negative: 84723
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007098 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 8248
[LightGBM] [Info] Number of data points in the train set: 148374, number of used features: 54
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.428990 -> initscore=-0.285972
[LightGBM] [Info] Start training from score -0.285972


/home/manas/Documents/assignments/ds/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Fold validation metrics:


,auc,pr_auc
model,,
cat,0.997703,0.996187
lgbm,0.996216,0.994444
xgb,0.996732,0.995157


## Train Meta-Learner Stacked Ensemble
We fit a logistic regression meta-learner on out-of-fold predictions to combine them optimally.

In [5]:
# Construct OOF predictions DataFrame
oof_val_df = pd.DataFrame({k: oof_probs[k] for k in models.keys()})
# Align with non-zero indices across folds (exclude warm-up phase)
val_idx = np.concatenate([te for _, _, te in folds])

X_meta = oof_val_df.iloc[val_idx]
y_meta = y_all[val_idx]

meta_clf = LogisticRegression(class_weight='balanced', random_state=SEED)
meta_clf.fit(X_meta, y_meta)
print(f"Meta-learner weights: {meta_clf.coef_}")

oof_blend = meta_clf.predict_proba(oof_val_df)[:, 1]

# Evaluate blend on validation indexes
blend_auc = roc_auc_score(y_meta, oof_blend[val_idx])
print(f"Stacked Ensemble AUC: {blend_auc:.5f}")

Meta-learner weights: [[2.20857707 2.12404802 6.12894568]]


Stacked Ensemble AUC: 0.99789


## Fit Calibrated Ensemble & Save Champion
We fit the full pipeline on the entire training set and calibrate using CalibratedClassifierCV.

In [6]:
# Build a Custom Stacking Classifier pipeline
from src.feature_engineering import StackingClassifierPipeline
stack_clf = StackingClassifierPipeline(models, meta_clf)
stack_clf.fit(X_all, y_all)

# Calibrate stacked model
from sklearn.calibration import FrozenEstimator
cal_clf = CalibratedClassifierCV(estimator=FrozenEstimator(stack_clf), method='sigmoid')
cal_clf.fit(X_all, y_all)

# Save model artifacts
joblib.dump(cal_clf, ART / 'final_calibrated_hgb_model.joblib')
pd.DataFrame({'feature': features}).to_csv(ART / 'final_model_features.csv', index=False)
print("Ensemble model and features exported.")

[LightGBM] [Info] Number of positive: 75565, number of negative: 103849
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.011654 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 8180
[LightGBM] [Info] Number of data points in the train set: 179414, number of used features: 54
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.421177 -> initscore=-0.317945
[LightGBM] [Info] Start training from score -0.317945


/home/manas/Documents/assignments/ds/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


/home/manas/Documents/assignments/ds/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


/home/manas/Documents/assignments/ds/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


/home/manas/Documents/assignments/ds/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


/home/manas/Documents/assignments/ds/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


/home/manas/Documents/assignments/ds/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Ensemble model and features exported.
